In [ ]:
# -*- coding: utf-8 -*-

"""

Created on Wed May 19 11:30:51 2021



@author: PristerM

"""
"""
Updated on Jan 6 2023
"""
"""
Updated on Aug 15 2025
Jingyi.W
"""

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os



print("Running FI FINFSA Web Scraping Tool v.1.0")

now=datetime.datetime.now()
filename= 'FI FINFSA data {}.xlsx'.format(str(now).replace(":",".")[:-7])


scriptfolder=os.path.dirname(os.path.abspath(__file__))
#scriptfolder=os.getcwd()
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process


os.chdir(scriptfolder)
#print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))
chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
		"plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

processdate=now.strftime('%Y-%m-%d')

driver.get('https://www.finanssivalvonta.fi/en/registers/supervised-entities/')

sleep(10)

try:
    driver.find_element(By.XPATH, '//button[contains(text(),"Accept all")]').click()
except:
    print('FI FINFSA - cookies button not found or failed')
driver.find_element(By.ID, 'tree-search-button').click()
sleep(10)
driver.find_element(By.ID, 'tree-export-button').click()
sleep(10)



csv_file=os.path.join(tempfolder, os.listdir(tempfolder)[0])
df=pd.read_csv(csv_file,sep=';',dtype=str)
os.remove(csv_file)
df.fillna('',inplace=True)
#print(df.columns)


df=df[['Company name', 'Business Identity Code', 'Contact information',

       'WWW-address']]

df.reset_index(drop=True, inplace=True)

contact=df['Contact information'].tolist()

for row in contact:

    if 'Finland' in row:

        sqldict['Cntry'].append('FI')

    else:

        sqldict['Cntry'].append('')

    if 'Telephone:' not in row and 'Telefax:' not in row:

        sqldict['Address_1'].append(row)

    else:

        row=row.replace('Telephone','**Telephone').replace('Telefax','**Telefax').split('**')

        row=[ele.strip() for ele in row if len(ele.strip())>0]

        if ',' in row[-1]:

            address=row[-1].split(',',1)[1].strip()

            phone_fax=row[-1].split(',',1)[0].strip()

            if 'Telephone' in phone_fax:

                sqldict['Phone'].append(phone_fax.split(':',1)[1].strip())

            else: 

                sqldict['Fax'].append(phone_fax.split(':',1)[1].strip())

            if len(row)==2:

                phone_fax=row[0]

                if 'Telephone' in phone_fax:

                    sqldict['Phone'].append(phone_fax.split(':',1)[1].strip())

                else: 

                    sqldict['Fax'].append(phone_fax.split(':',1)[1].strip())

                

            sqldict['Address_1'].append(address)

        else:

            sqldict['Address_1'].append("")

    for key in ['Phone', 'Fax']:

        if len(sqldict[key])<len(sqldict['Address_1']):

            sqldict[key].append("")

            

sqldict['Name'].extend(df['Company name'].tolist())

sqldict['InternalID_1'].extend(df['Business Identity Code'].tolist())

sqldict['Website'].extend(df['WWW-address'].tolist())



for times in range(len(sqldict['Name'])):
    sqldict["ListProcessDate"].append(processdate)  
    sqldict['ListCode'].append("1")		
    sqldict["RegulationType"].append('Supervised')                
    sqldict["RegCtry"].append("FI")                
    sqldict["RegCode"].append("FINFSA")   
    sqldict["InternalID_1_type"].append("Business Identity Code")   
    for key in sqldict.keys():                 
        if len(sqldict[key])<len(sqldict["ListProcessDate"]):
            sqldict[key].append("")

for key in sqldict.keys(): 
    print(key,len(sqldict[key]))


df=pd.DataFrame(sqldict)
writer = ExcelWriter(filename)
df.to_excel(writer, 'SQL Ready', index=False)


writer.save()
writer.close()

driver.quit()

    
    